# 01. Генерация датасета

Реальные логи платформы недоступны, поэтому воронка и монетизация моделируются заданными вероятностями переходов между шагами и распределениями сумм платежей.

Что важно для дальнейшего анализа:

- вероятности используются только в этом ноутбуке. В ноутбуках 03 и 04 они не применяются, все выводы получены из данных, а не из параметров генератора;
- значения генератора случайных чисел зафиксированы, база пересобирается идентично;
- ноутбук идемпотентен: таблицы пересоздаются, повторный запуск не приводит к дублированию данных.

В модель заложены три отклонения, которые предстоит обнаружить анализом: провал на втором шаге онбординга, пониженная доходимость с мобильных устройств и планшетов, тяжёлый хвост крупных сделок в выручке.

Схема данных: `users` → `user_events` → `transactions`.

In [1]:
import random
import sqlite3
from contextlib import closing
from datetime import datetime, timedelta
from itertools import count
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
DB_PATH = Path('../data/skillforge.db')
N_USERS = 25_000
PERIOD_START = datetime(2026, 1, 1)
PERIOD_END = datetime(2026, 6, 30)

RANDOM_SEED = 42

## Пользователи

Регистрации распределены равномерно по первому полугодию 2026 года. Каждому пользователю присваиваются три атрибута, известные в момент регистрации: канал привлечения, устройство и страна. Только они используются далее как доэкспериментальные признаки, поскольку иных данных о пользователе на момент регистрации нет.

In [3]:
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

period_seconds = int((PERIOD_END - PERIOD_START).total_seconds())
registration_dts = sorted(
    PERIOD_START + timedelta(seconds=int(offset))
    for offset in np.random.randint(0, period_seconds, size=N_USERS)
)

df_users = pd.DataFrame({
    'user_id': range(10_001, 10_001 + N_USERS),
    'registration_dt': [dt.strftime('%Y-%m-%d %H:%M:%S') for dt in registration_dts],
    'traffic_channel': np.random.choice(
        ['organic', 'paid_search', 'influencers', 'targeted_ads'],
        size=N_USERS, p=[0.35, 0.30, 0.15, 0.20]),
    'device_type': np.random.choice(
        ['mobile', 'desktop', 'tablet'],
        size=N_USERS, p=[0.55, 0.38, 0.07]),
    'country': np.random.choice(
        ['RU', 'KZ', 'BY', 'UZ', 'Other'],
        size=N_USERS, p=[0.70, 0.15, 0.07, 0.05, 0.03]),
})

df_users.head()

,user_id,registration_dt,traffic_channel,device_type,country
0,10001,2026-01-01 00:00:37,paid_search,desktop,RU
1,10002,2026-01-01 00:15:17,targeted_ads,mobile,KZ
2,10003,2026-01-01 00:27:38,organic,desktop,RU
3,10004,2026-01-01 00:53:16,paid_search,desktop,BY
4,10005,2026-01-01 00:53:32,influencers,desktop,RU


In [4]:
P_START_ONBOARDING = {'mobile': 0.88, 'desktop': 0.88, 'tablet': 0.75}
P_COMPLETE_STEP_1 = {'organic': 0.82, 'paid_search': 0.82, 'influencers': 0.82, 'targeted_ads': 0.76}
P_COMPLETE_STEP_2 = {'desktop': 0.70, 'mobile': 0.52, 'tablet': 0.45}
P_COMPLETE_STEP_3 = 0.85
P_COMPLETE_ONBOARDING = 0.90
P_VIEW_CATALOG = {True: 0.65, False: 0.25}
P_START_TRIAL = 0.60
P_FINISH_TRIAL = 0.70
P_PAYWALL_AFTER_TRIAL = 0.75
P_PAYWALL_WITHOUT_TRIAL = 0.30

EXTRA_CATALOG_VIEWS = ([0, 1, 2], [0.55, 0.32, 0.13])
EXTRA_PAYWALL_VIEWS = ([0, 1], [0.70, 0.30])

events = []
event_ids = count(1_000_001)


def log_event(user_id, event_name, event_dt):
    events.append((next(event_ids), user_id, event_name, event_dt.strftime('%Y-%m-%d %H:%M:%S')))


for user_id, registration_dt, channel, device in zip(
        df_users['user_id'], df_users['registration_dt'],
        df_users['traffic_channel'], df_users['device_type']):

    reg_dt = datetime.strptime(registration_dt, '%Y-%m-%d %H:%M:%S')

    log_event(user_id, 'view_landing', reg_dt - timedelta(minutes=random.randint(3, 20)))
    log_event(user_id, 'start_registration', reg_dt - timedelta(minutes=random.randint(1, 3)))
    log_event(user_id, 'complete_registration', reg_dt)

    current_dt = reg_dt
    completed_onboarding = False

    if random.random() < P_START_ONBOARDING[device]:
        current_dt += timedelta(seconds=random.randint(10, 60))
        log_event(user_id, 'start_onboarding', current_dt)

        if random.random() < P_COMPLETE_STEP_1[channel]:
            current_dt += timedelta(seconds=random.randint(20, 120))
            log_event(user_id, 'complete_step_1', current_dt)

            if random.random() < P_COMPLETE_STEP_2[device]:
                current_dt += timedelta(seconds=random.randint(30, 200))
                log_event(user_id, 'complete_step_2', current_dt)

                if random.random() < P_COMPLETE_STEP_3:
                    current_dt += timedelta(seconds=random.randint(20, 90))
                    log_event(user_id, 'complete_step_3', current_dt)

                    if random.random() < P_COMPLETE_ONBOARDING:
                        current_dt += timedelta(seconds=random.randint(10, 45))
                        log_event(user_id, 'complete_onboarding', current_dt)
                        completed_onboarding = True

    if random.random() < P_VIEW_CATALOG[completed_onboarding]:
        current_dt += timedelta(minutes=random.randint(2, 60))
        log_event(user_id, 'view_catalog', current_dt)

        for _ in range(np.random.choice(EXTRA_CATALOG_VIEWS[0], p=EXTRA_CATALOG_VIEWS[1])):
            current_dt += timedelta(minutes=random.randint(5, 180))
            log_event(user_id, 'view_catalog', current_dt)

        if random.random() < P_START_TRIAL:
            current_dt += timedelta(minutes=random.randint(1, 15))
            log_event(user_id, 'start_trial_lesson', current_dt)

            if random.random() < P_FINISH_TRIAL:
                current_dt += timedelta(minutes=random.randint(15, 45))
                log_event(user_id, 'finish_trial_lesson', current_dt)

                if random.random() < P_PAYWALL_AFTER_TRIAL:
                    current_dt += timedelta(minutes=random.randint(1, 10))
                    log_event(user_id, 'view_paywall', current_dt)

                    for _ in range(np.random.choice(EXTRA_PAYWALL_VIEWS[0], p=EXTRA_PAYWALL_VIEWS[1])):
                        current_dt += timedelta(hours=random.randint(1, 48))
                        log_event(user_id, 'view_paywall', current_dt)

        elif random.random() < P_PAYWALL_WITHOUT_TRIAL:
            current_dt += timedelta(minutes=random.randint(2, 20))
            log_event(user_id, 'view_paywall', current_dt)

df_events = pd.DataFrame(events, columns=['event_id', 'user_id', 'event_name', 'event_dt'])

print(f'Событий: {len(df_events):,} | типов: {df_events["event_name"].nunique()}')
df_events.head()

Событий: 170,219 | типов: 12


,event_id,user_id,event_name,event_dt
0,1000001,10001,view_landing,2025-12-31 23:54:37
1,1000002,10001,start_registration,2025-12-31 23:59:37
2,1000003,10001,complete_registration,2026-01-01 00:00:37
3,1000004,10001,start_onboarding,2026-01-01 00:01:02
4,1000005,10001,complete_step_1,2026-01-01 00:02:56


## События

Поведение пользователя описывается цепочкой вложенных вероятностей: переход к шагу 2 возможен только после прохождения шага 1 и так далее. Вложенность намеренно нарушена в двух местах, что соответствует устройству реального продукта:

- каталог доступен всем, в том числе после незавершённого онбординга, но с меньшей вероятностью — 0.25 против 0.65;
- пейволл может быть показан без прохождения пробного урока.

Просмотр каталога и пейволла может повторяться: пользователь возвращается к ним неоднократно. Вследствие этого число событий у пользователя не совпадает с числом уникальных типов событий, и обе метрики в витрине несут различный смысл.

Отклонения модели: планшеты хуже стартуют (0.75 против 0.88), таргетированная реклама хуже проходит первый шаг (0.76 против 0.82), мобильные устройства заметно проваливают шаг 2 (0.52 против 0.70 на десктопе).

In [5]:
PRICE_GRID = {
    'monthly_sub': ([990, 1490, 1990, 2490], [0.2, 0.4, 0.3, 0.1]),
    'annual_sub': ([9900, 14900, 19900, 24900], [0.3, 0.4, 0.2, 0.1]),
    'one_time_course': ([3900, 5900, 8900, 12900], [0.4, 0.3, 0.2, 0.1]),
}

P_PAY_AFTER_PAYWALL = 0.22
P_PAY_WITHOUT_PAYWALL = 0.015
P_ENTERPRISE_DEAL = 0.012

last_event_dt = df_events.groupby('user_id')['event_dt'].max()
paywall_users = set(df_events.loc[df_events['event_name'] == 'view_paywall', 'user_id'])

transactions = []
transaction_ids = count(5_000_001)

for user_id in df_users['user_id']:
    conversion_chance = P_PAY_AFTER_PAYWALL if user_id in paywall_users else P_PAY_WITHOUT_PAYWALL
    if random.random() >= conversion_chance:
        continue

    payment_dt = (datetime.strptime(last_event_dt[user_id], '%Y-%m-%d %H:%M:%S')
                  + timedelta(minutes=random.randint(5, 120)))

    for _ in range(np.random.choice([1, 2, 3, 4], p=[0.72, 0.18, 0.07, 0.03])):
        if payment_dt > PERIOD_END:
            break

        if random.random() < P_ENTERPRISE_DEAL:
            subscription_type = 'annual_sub'
            amount = round(float(np.random.pareto(a=2.0) * 25_000 + 45_000), 2)
        else:
            subscription_type = np.random.choice(
                ['monthly_sub', 'annual_sub', 'one_time_course'], p=[0.62, 0.15, 0.23])
            price_values, price_weights = PRICE_GRID[subscription_type]
            amount = float(np.random.choice(price_values, p=price_weights))

        transactions.append((
            next(transaction_ids),
            user_id,
            payment_dt.strftime('%Y-%m-%d %H:%M:%S'),
            amount,
            subscription_type,
            np.random.choice(['success', 'failed', 'refunded'], p=[0.91, 0.07, 0.02]),
        ))

        payment_dt += timedelta(days=random.randint(25, 35))

df_transactions = pd.DataFrame(transactions, columns=[
    'transaction_id', 'user_id', 'payment_dt', 'amount', 'subscription_type', 'payment_status'])

print(f'Транзакций: {len(df_transactions):,} | платящих: {df_transactions["user_id"].nunique():,}')
df_transactions.head()

Транзакций: 1,614 | платящих: 1,239


,transaction_id,user_id,payment_dt,amount,subscription_type,payment_status
0,5000001,10002,2026-01-01 01:48:24,5900.0,one_time_course,success
1,5000002,10006,2026-01-01 02:32:03,5900.0,one_time_course,success
2,5000003,10006,2026-02-02 02:32:03,19900.0,annual_sub,success
3,5000004,10006,2026-02-28 02:32:03,1490.0,monthly_sub,success
4,5000005,10006,2026-04-04 02:32:03,1490.0,monthly_sub,success


## Транзакции

Основная развилка монетизации: пользователь, увидевший пейволл, совершает покупку с вероятностью 0.22, не увидевший — 0.015.

Отдельно моделируется B2B-хвост: 1.2% сделок получают сумму из распределения Парето от 45 000 ₽, отдельные значения достигают сотен тысяч. Такие сделки существенно искажают средние значения в аналитике подписочных продуктов, поэтому в ноутбуке 03 предусмотрена отдельная проверка: не обусловлена ли найденная разница между сегментами небольшим числом крупных платежей.

Статусы платежей неоднородны: 7% отказов и 2% возвратов. Во всех витринах выручка считается только по статусу `success`.

In [6]:
DDL = [
    'DROP TABLE IF EXISTS transactions;',
    'DROP TABLE IF EXISTS user_events;',
    'DROP TABLE IF EXISTS users;',
    '''
    CREATE TABLE users (
        user_id INTEGER PRIMARY KEY,
        registration_dt TEXT NOT NULL,
        traffic_channel TEXT NOT NULL,
        device_type TEXT NOT NULL,
        country TEXT NOT NULL
    );''',
    '''
    CREATE TABLE user_events (
        event_id INTEGER PRIMARY KEY,
        user_id INTEGER NOT NULL,
        event_name TEXT NOT NULL,
        event_dt TEXT NOT NULL,
        FOREIGN KEY (user_id) REFERENCES users(user_id) ON DELETE CASCADE
    );''',
    '''
    CREATE TABLE transactions (
        transaction_id INTEGER PRIMARY KEY,
        user_id INTEGER NOT NULL,
        payment_dt TEXT NOT NULL,
        amount REAL NOT NULL,
        subscription_type TEXT NOT NULL,
        payment_status TEXT NOT NULL,
        FOREIGN KEY (user_id) REFERENCES users(user_id) ON DELETE CASCADE
    );''',
    'CREATE INDEX idx_users_reg_dt ON users(registration_dt);',
    'CREATE INDEX idx_events_user_dt ON user_events(user_id, event_dt);',
    'CREATE INDEX idx_events_name ON user_events(event_name);',
    'CREATE INDEX idx_tx_user_dt ON transactions(user_id, payment_dt);',
]

DB_PATH.parent.mkdir(parents=True, exist_ok=True)

with closing(sqlite3.connect(DB_PATH)) as conn:
    conn.execute('PRAGMA foreign_keys = ON;')
    for statement in DDL:
        conn.execute(statement)

    df_users.to_sql('users', conn, if_exists='append', index=False)
    df_events.to_sql('user_events', conn, if_exists='append', index=False)
    df_transactions.to_sql('transactions', conn, if_exists='append', index=False)
    conn.commit()

    loaded = pd.read_sql_query('''
        SELECT 'users' AS table_name, COUNT(*) AS rows FROM users
        UNION ALL SELECT 'user_events', COUNT(*) FROM user_events
        UNION ALL SELECT 'transactions', COUNT(*) FROM transactions;
    ''', conn)

print(f'База пересобрана: {DB_PATH.resolve()}')
loaded

База пересобрана: /Users/timofejgavrikov/Documents/skillforgeAB/data/skillforge.db


,table_name,rows
0,users,25000
1,user_events,170219
2,transactions,1614


## Запись в SQLite

Таблицы пересоздаются, внешние ключи и индексы объявлены явно. Индексы построены по связкам, используемым в последующих группировках: `user_id + event_dt` для событий и платежей, `event_name` для фильтрации по типу события.